## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [3]:
# Installation for GPU llama-cpp-python
# uncomment and run the following code in case GPU is being used
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q

# Installation for CPU llama-cpp-python
# uncomment and run the following code in case GPU is not being used
# !CMAKE_ARGS="-DLLAMA_CUBLAS=off" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 96.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 284.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 275.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 278.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.21.0 requires opentelemetry-api<=1.37.0,>=1.37.0, but you have opentelemetry-api 1.39.1 which is incompatible.
google-adk 1.21.0 requires opentelemetry-sdk<=1.37.0,>=1.37.0, but you have opentelemetry-sdk 1.39.1 which is incompatible.
langgraph-prebuilt 1.0.5 requires langchain-core>=1.0.0, but you have langchain-

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [4]:
# For installing the libraries & downloading models from HF Hub
#!pip install huggingface_hub==0.35.3 pandas==2.2.2 tiktoken==0.12.0 pymupdf==1.26.5 langchain==0.3.#27 langchain-community==0.3.31 chromadb==1.1.1 sentence-transformers==5.1.1 numpy==2.3.3 -q

!pip install huggingface_hub==0.35.3 pandas==2.2.2 tiktoken==0.12.0 pymupdf==1.26.5 langchain==0.3.27 langchain-community==0.3.31 chromadb==1.1.1 sentence-transformers "numpy<2.0.0" -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
pytensor 2.35.1 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.


**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [1]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

## Question Answering using LLM

#### Downloading and Loading the model

In [2]:
model_name_or_path = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
model_basename = "mistral-7b-instruct-v0.2.Q6_K.gguf"

model_name_or_path_llama = "TheBloke/Llama-2-7B-Chat-GGUF"
model_basename_llama = "llama-2-7b-chat.Q6_K.gguf"

#model_2_name_or_path = "MaziyarPanahi/Mistral-7B-Instruct-v0.3-GGUF"
#model_2_basename = "Mistral-7B-Instruct-v0.3.IQ1_M.gguf"

In [3]:
model_path = hf_hub_download(repo_id=model_name_or_path, filename=model_basename)

model_path_llama = hf_hub_download(repo_id=model_name_or_path_llama, filename=model_basename_llama)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [6]:
llm = Llama(
    model_path=model_path,
    n_ctx=4096,
    n_gpu_layers=32,
    n_batch = 512
)

AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


#### Response

In [4]:
def response(query,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return model_output['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [7]:
user_input = "What is the protocol for managing sepsis in a critical care unit?"
print(response(user_input))



Sepsis is a life-threatening condition that can arise from an infection, and it requires prompt recognition and aggressive management in a critical care unit. The following are general steps for managing sepsis in a critical care unit:

1. Early recognition: Recognize the signs and symptoms of sepsis early and initiate treatment as soon as possible. Sepsis can present with various clinical features, including fever or hypothermia, tachycardia or bradycardia, altered mental status, respiratory distress, and lactic acidosis.
2. ABCs


### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [8]:
user_input = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
print(response(user_input))

Llama.generate: prefix-match hit




Appendicitis is a medical condition characterized by inflammation of the appendix, a small pouch-like structure that extends from the large intestine. The symptoms of appendicitis can vary from person to person, but some common signs include:

1. Abdominal pain: The pain is typically located in the lower right side of the abdomen and may be constant or come and go. It may start as a mild discomfort that worsens over time.
2. Loss of appetite: People with appendicitis often lose their appetite due to abdominal pain and nausea


### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [9]:
user_input = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
print(response(user_input))

Llama.generate: prefix-match hit




Sudden patchy hair loss, also known as alopecia areata, is a common autoimmune disorder that affects the hair follicles. It can result in round or oval bald patches on the scalp, but it can also occur on other parts of the body such as the beard area, eyebrows, and eyelashes.

The exact cause of alopecia areata is not known, but it's believed to be related to a problem with the immune system. Some possible triggers for this condition include stress, genetics, viral infections, and certain medications.


### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [10]:
user_input="What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
print(response(user_input))

Llama.generate: prefix-match hit




A person who has sustained a physical injury to the brain tissue may require various treatments depending on the severity and location of the injury. Here are some common treatments that may be recommended:

1. Emergency care: In case of a traumatic brain injury (TBI), it is essential to seek emergency medical attention as soon as possible. The primary goal of emergency care is to prevent further damage to the brain, stabilize vital signs, and manage any life-threatening conditions.
2. Medications: Depending on the symptoms, healthcare professionals may prescribe medications to manage various conditions associated with a


### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [11]:
user_input="What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
print(response(user_input))

Llama.generate: prefix-match hit




First and foremost, if you suspect that someone has fractured their leg while hiking, it's essential to ensure their safety and prevent further injury. Here are some necessary precautions:

1. Keep the person calm and still: Encourage them to remain as still as possible to minimize pain and prevent worsening the injury.
2. Assess the situation: Check for any signs of shock, such as pale skin, rapid heartbeat, or shallow breathing. If you notice these symptoms, seek medical help immediately.
3. Immobilize the leg: Use a splint, sl


**OBSERVATIONS**:

---
- The answers are not complete. Due to the max_tokens, the response stops abruptly.
- In the **first** question, The response does not mention what ABCs mean. Issue: Response is not summarized well.
- In the **second** question, only the first part of the question is answered. The middle or the last part of the question is not covered in the response. Issue: Incomplete response.
- The **third** question's response seems good enough, although it considers 1 case
- The **fourth** and **fifth** question's response is incomplete
- Let us address the issues in the nect section using prompt engineering and tweaking some parameters.




## Question Answering using LLM with Prompt Engineering

In [12]:
system_prompt = """Role: You are an AI Healthcare Specialist who has knowledge through vast volumes of medical data.
Goal and Guidelines: Your primary goal is :
- To assist the medical professionals in answering the questions asked to you as input.
- To provide evidence-based, accurate, and relevant information regarding the questions asked to you as input.

- Formulate the answers based on only the knowledge you have been provided for training to help deliver accurate and timely diagnoses,
while prioritizing patient safety.
- Summarize the answers without losing critical information.
- Ensure the answers are complete, irrespective of the max_tokens

- Context: "The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.
Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.
To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

My goal as: As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to understand issues like information overload, apply AI techniques to streamline decision-making, analyze its impact on diagnostics and patient outcomes, evaluate its potential to standardize care practices, and create a functional prototype demonstrating its feasibility and effectiveness."


- If the answer to the question cannot be found, do not hallucinate.
- If answer is not found, Reply: 'Cannot provide information'
"""

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [10]:
user_input_1 = system_prompt+"\n"+ "What is the protocol for managing sepsis in a critical care unit?"
print(response(user_input_1, max_tokens=200,temperature=0,top_p=0.90,top_k=100 ))



Answer: Sepsis is a life-threatening condition caused by the body's response to infection. In a critical care unit, managing sepsis involves several steps to ensure optimal patient outcomes. Here are some key aspects of sepsis management in a critical care setting:

1. Early recognition and diagnosis: Recognizing sepsis early is crucial for effective treatment. Look out for signs such as fever, chills, rapid heart rate, rapid breathing, confusion, and low blood pressure. Use the Sequential Organ Failure Assessment (SOFA) score to evaluate organ dysfunction.

2. Fluid resuscitation: Administer intravenous fluids to maintain adequate blood pressure and improve tissue perfusion. Monitor fluid intake and output closely to avoid overloading the patient.

3. Antibiotic therapy: Start broad-spectrum antibiotics as soon as possible based on clinical suspicion and culture results.


**OBSERVATIONS**:


---

- Since the response seems incomplete, let's increase the max_tokens in the next question and also test with varying the parameter k.
- We will compare these responses with the same question's response visually from the previous section to see if there is any difference.



### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [19]:
user_input_2 = system_prompt+"\n"+ "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
print(response(user_input_2, max_tokens=300,temperature=0,top_p=0.90,top_k=120))

Llama.generate: prefix-match hit




Answer: Appendicitis is a medical condition characterized by inflammation of the appendix, a small pouch that extends from the large intestine. Common symptoms include abdominal pain, usually starting around the navel and then moving to the lower right side, loss of appetite, nausea, vomiting, fever, and constipation or diarrhea.

Appendicitis cannot be cured via medicine alone due to the risk of appendix rupture, which can lead to severe complications such as peritonitis. If diagnosed early, a surgical procedure called an appendectomy is typically recommended. This involves removing the inflamed appendix to prevent further complications and promote healing. The most common type of appendectomy is laparoscopic, which uses small incisions and a camera to guide the surgeon's instruments. Open appendectomies are less common but may be necessary in certain cases.

Regardless of the surgical approach, it is essential that patients receive prompt treatment to minimize complications and imp

**OBSERVATIONS**:


---
- The response was still incomplete at max_tokens=256, so increased it to 300. Post that, it was complete. Based on the questions, it seems valid that it is would lose context in such few tokens.
- The response varies a bit due to changing the parameters of k and thus increasing the text generalisation. Hence it is able to answer for all 3 sub questions in the questions.
- The response is complete and summarized well. Now let's change the temperature for next run and still increase max tokens.


### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [20]:
user_input_3 = system_prompt+"\n"+ "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
print(response(user_input_3, max_tokens=512,temperature=0.2,top_p=0.90,top_k=120))

Llama.generate: prefix-match hit




Answer: Sudden patchy hair loss, also known as alopecia areata, is an autoimmune disorder that results in hair loss from some or all areas of the scalp. The exact cause of this condition is unknown, but it's believed to be related to a problem with the immune system.

Effective treatments for addressing sudden patchy hair loss include:
1. Corticosteroids: These medications can help reduce inflammation and suppress the immune response that causes hair loss. They can be applied topically or taken orally.
2. Immunotherapy: This treatment involves using substances that stimulate an immune response to promote hair regrowth. One such substance is diphenylcyclopropenone (DPCP).
3. Minoxidil: A medication applied directly to the scalp, minoxidil can help slow down or stop hair loss and promote new hair growth.
4. Hair transplantation: In severe cases where large areas of the scalp are affected, hair transplantation may be an option. This involves taking healthy hair follicles from one area o

**OBSERVATIONS**:


---
- By changing the temperature and top_p values, we notice a difference in the response.
- Increasing the max_tokens has provided a more complete response.
- Let's change the top_p for next question



### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [21]:
user_input_4 = system_prompt+"\n"+ "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
print(response(user_input_4, max_tokens=512,temperature=0.2,top_p=0.85,top_k=120))

Llama.generate: prefix-match hit




Answer: A person with a brain injury may require various treatments depending on the severity and location of the injury. Some common treatments include:

1. Medications: To manage symptoms such as pain, seizures, or swelling in the brain. For instance, corticosteroids might be prescribed to reduce inflammation.
2. Rehabilitation therapy: Physical, occupational, and speech therapies can help patients regain lost skills and improve overall function. This may include exercises for strength, coordination, balance, and cognitive abilities.
3. Surgery: In some cases, surgery might be necessary to remove blood clots or repair damaged tissue. Craniotomy, a surgical procedure where a portion of the skull is removed, can help relieve pressure on the brain.
4. Assistive devices: Devices like wheelchairs, braces, or communication aids can help patients manage their daily activities and improve their quality of life.
5. Lifestyle modifications: Adopting healthy habits such as regular exercise, a

**OBSERVATIONS**:


---
- The response seems to be complete, relevant and well structured.
- top_p decrease helps cover more probabilities and more use cases. We can keep it high if our question might need a very specific response and low, when we might need more options to consider.



### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [22]:
user_input_5 = system_prompt+"\n"+ "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
print(response(user_input_5, max_tokens=512,temperature=0.1,top_p=0.95,top_k=120))

Llama.generate: prefix-match hit




Answer: A fractured leg, also known as a broken leg, is a common injury that occurs due to high-impact falls or twisting motions. The necessary precautions and treatment steps for a person with a fractured leg are as follows:

1. Immobilization: The first step is to immobilize the affected leg using a splint, cast, or brace to prevent further damage and promote healing. This will also help in reducing pain and swelling.

2. Pain management: Over-the-counter pain medications like acetaminophen or ibuprofen can be used to manage pain. In severe cases, prescription painkillers may be required.

3. Rest and elevation: The patient should rest the leg as much as possible and keep it elevated above heart level to minimize swelling and promote proper blood flow.

4. Proper positioning: Ensure that the patient is positioned comfortably with adequate support for their back, head, and injured leg.

5. Monitor vital signs: Keep a close eye on the patient's pulse, respiration rate, and temperatur

**OBSERVATIONS**:


---
- This seems to be a good combination of parameters for our use case to get response using prompt engineering based on our question. This exercise helped to see how changing parameters changes the responses.
- Let's experiment with RAG next.



## Data Preparation for RAG

### Loading the Data

In [13]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
manual_pdf_path = "/content/drive/My Drive/AIML/Project5/medical_diagnosis_manual.pdf"

In [15]:
pdf_loader = PyMuPDFLoader(manual_pdf_path)

In [16]:
manual = pdf_loader.load()

### Data Overview

#### Checking the first 5 pages

In [17]:
for i in range(5):
    print(f"Page Number : {i+1}",end="\n")
    print(manual[i].page_content,end="\n")

Page Number : 1
riddhic17@gmail.com
8I4KUSQZ5M
meant for personal use by riddhic17@gm
shing the contents in part or full is liable
Page Number : 2
riddhic17@gmail.com
8I4KUSQZ5M
This file is meant for personal use by riddhic17@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 3
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    ...........................................................................................................................................................................................
53
1 - 

#### Checking the number of pages

In [18]:
len(manual)

4114

**OBSERVATIONS**:


---
- There are ~4114 pages in the manual
- There are about 144 chapters, The first ~53 pages or so might be introduction and other content and actual sections and chapters with useful information starts after that.
- Let's split the document in chunks based on this info.







### Data Chunking

In [19]:
text_splitter_2000 = RecursiveCharacterTextSplitter.from_tiktoken_encoder(encoding_name ='cl100k_base',chunk_size=2000,chunk_overlap=400)
document_chunks_2000 = text_splitter_2000.split_documents(manual)

In [20]:
text_splitter_1000 = RecursiveCharacterTextSplitter.from_tiktoken_encoder(encoding_name ='cl100k_base',chunk_size=1000,chunk_overlap=250)
document_chunks_1000 = text_splitter_1000.split_documents(manual)

In [21]:
text_splitter_500 = RecursiveCharacterTextSplitter.from_tiktoken_encoder(encoding_name ='cl100k_base',chunk_size=500,chunk_overlap=100)
document_chunks_500 = text_splitter_500.split_documents(manual)

In [22]:
print(len(document_chunks_2000),'\n',len(document_chunks_1000),'\n',len(document_chunks_500))

4114 
 4685 
 9335


In [23]:
print(document_chunks_2000[67].page_content)

absorption. Most patients receiving phenytoin, phenobarbital, primidone, or phenothiazines develop folate
deficiency, probably because hepatic microsomal drug-metabolizing enzymes are affected. Folate
supplements may
[Table 1-6. Effects of Some Drugs on Nutrition]
make phenytoin less effective. Anticonvulsants can cause vitamin D deficiency. Malabsorption of vitamin
B12 can occur with use of aminosalicylic acid, slow-release K iodide, colchicine, trifluoperazine, ethanol,
and oral contraceptives. Oral contraceptives with a high progestin dose can cause depression, probably
because of metabolically induced tryptophan deficiency.
Food Additives and Contaminants
Additives: Chemicals are often combined with foods to facilitate their processing and preservation or to
enhance their desirability. Only amounts of additives shown to be safe by laboratory tests are permitted in
commercially prepared foods.
Weighing the benefits of additives (eg, reduced waste, increased variety of available food

**OBSERVATIONS**:


---
- After printing some random chunks, Seems like chapter 1 starts from end of document_chunk:67
- Based on our previous questions to the LLM, let's try to print some chunks to check where actual answers may be extracted from. This is just for experimenting.
- Also, We can test chunk overlap and use these chunks for vector similarity search


In [24]:
print(document_chunks_2000[1045].page_content)


Amyloidosis is suspected clinically but can be diagnosed only by biopsy. Subcutaneous abdominal fat pad
aspiration and biopsy of rectal mucosa are the best approaches. Other useful biopsy sites are the
gingiva, skin, nerves, kidneys, and liver. Tissue sections are stained with Congo red dye and examined
with a polarizing microscope for characteristic birefringence. Isotopically labeled serum AP (in which AP
represents the pentagonal component of amyloid) can be used in a scintigraphic test to confirm the
diagnosis.
Prognosis
Prognosis depends on the type of amyloidosis and the organ system involved. AL amyloidosis with
multiple myeloma has the poorest prognosis: death within 1 yr is common. Untreated ATTR amyloidoses
are fatal within 10 to 15 yr. Prognosis in other familial amyloidoses varies. In general, renal or cardiac
involvement in patients with any type of amyloidosis is of particular concern.
Prognosis in AA amyloidosis depends on successful treatment of the underlying disorder,

In [25]:
print(document_chunks_1000[1198].page_content)


Indications for screening patients < 20 yr are atherosclerotic risk factors, such as diabetes, hypertension,
cigarette smoking, and obesity; premature CAD in a parent, grandparent, or sibling; or a cholesterol level
> 240 mg/dL (> 6.2 mmol/L) or known dyslipidemia in a parent. If information on relatives is unavailable,
as in the case of adopted children, screening is at the discretion of the health care practitioner.
Patients with an extensive family history of heart disease should also be screened by measuring Lp(a)
levels.
Treatment
• Risk assessment by explicit criteria
• Lifestyle changes (eg, exercise, dietary modification)
• For high LDL cholesterol, statins, sometimes bile acid sequestrants, ezetimibe, and other measures
• For high TG or low HDL cholesterol, niacin, fibrates, and sometimes other measures
General principles: Treatment is indicated for all patients with cardiovascular disease (secondary
prevention) and for some without (primary prevention). The National Institute

In [37]:
print(document_chunks_1000[1199].page_content)

ATPIII guidelines recommend that these patients have LDL cholesterol levels lowered to < 100 mg/dL, but
accumulating evidence
[Table 100-5. Ncep Adult Treatment Panel III Guidelines for Treatment of Hyperlipidemia]
suggests that this target may be too high and a target LDL cholesterol < 70 mg/dL is an option for
patients at very high risk (eg, patients with known CAD and diabetes, other poorly controlled risk factors,
metabolic syndrome, or acute coronary syndrome). When drugs are used, a dose providing at least a 30
to 40% decrease in LDL cholesterol is desirable (see
Table 100-8).
For children, the AAP recommends dietary treatment for children with LDL cholesterol > 110 mg/dL. Drug
therapy is recommended for children > 8 yr and with either of the following:
• Poor response to dietary therapy, LDL cholesterol ≥ 190 mg/dL, and no family history of premature
cardiovascular disease
• LDL cholesterol ≥ 160 mg/dL and a family history of premature cardiovascular disease or ≥ 2 risk factors


In [38]:
document_chunks_500[9011].page_content.split("\n")

['private insurance 3163, 3475',
 'Medical malpractice 3473',
 'Medicare 3155-3161, 3157',
 'Part A 3156-3159',
 'Part B 3159-3160',
 'Part C 3160',
 'Part D 3160-3161, 3161',
 'Medicare Advantage Plans 3160',
 'Medicolegal issues 3468-3473',
 'Medigap 3163',
 'Meditation 3414, 3417',
 'Mediterranean anemia 946-947',
 'Mediterranean fever 1246-1247',
 'familial 3026-3027',
 'Mediterranean spotted fever 1281, 1282',
 'Medium-chain acyl dehydrogenase deficiency 3018',
 'Medoidioplasty 1570',
 'Medroxyprogesterone',
 'for contraception 2583, 2588-2589',
 'in dysfunctional uterine bleeding 2512',
 'in endometriosis 2540',
 'in hot flushes 2520',
 'in pedophilia 1573',
 'in protein-energy undernutrition 18',
 'tremors and 1775',
 'in uterine fibroids 2537',
 'Medullary cystic disease 2382, 2384-2385',
 'Medullary sponge kidney 2383-2384',
 'Medulloblastoma 1814, 1817-1818',
 'in children 2941-2942',
 'MedWatch 3185',
 "Mees' lines 733",
 'Mefloquine 1384, 1387, 1387, 1388',
 'Megacolon',
 '

In [39]:
document_chunks_500[9012].page_content.split("\n")

['Chagas 1377',
 'Megaloblastic anemia 932-933 (see also Anemia, megaloblastic)',
 'oral lesions in 495',
 'vitamin B12 deficiency and 28, 38, 922, 932',
 'Megestrol acetate',
 'in cancer chemotherapy 1068',
 'in hot flushes 2520',
 'in protein-energy undernutrition 18',
 'in uterine fibroids 2537',
 'Meglumine antimonate 1380',
 'Meibomian glands',
 'dysfunction of 575',
 'infection of 576-577',
 'occlusion of 576-577',
 "Meige's disease 1762, 1997, 2235",
 "Meigs' syndrome 2568",
 'Meiosis 3373',
 'Melancholy 1541',
 'The Merck Manual of Diagnosis & Therapy, 19th Edition',
 'M',
 '3937',
 'riddhic17@gmail.com',
 '8I4KUSQZ5M',
 'This file is meant for personal use by riddhic17@gmail.com only.',
 'Sharing or publishing the contents in part or full is liable for legal action.']

**OBSERVATIONS**:


---

- Based on our parameters, there is overlap in the chunks comparing the document_chunks_500 of pages 9011 and 9012.
- The chunk sizes match our parameters



### Embedding

In [26]:
embedding_model = SentenceTransformerEmbeddings(model_name="BAAI/bge-base-en-v1.5")

/tmp/ipython-input-2212284707.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = SentenceTransformerEmbeddings(model_name="BAAI/bge-base-en-v1.5")


In [27]:
embedding_1 = embedding_model.embed_query(document_chunks_500[0].page_content)
embedding_2 = embedding_model.embed_query(document_chunks_500[1].page_content)

In [28]:
print("Dimension of the embedding vector ",len(embedding_1))
len(embedding_1)==len(embedding_2)

Dimension of the embedding vector  768


True

In [29]:
embedding_1,embedding_2

([0.026973985135555267,
  -0.010534685105085373,
  0.04116728901863098,
  0.04706408455967903,
  0.03850699961185455,
  0.004406283609569073,
  0.012045314535498619,
  0.005263103637844324,
  0.0177164226770401,
  -0.01748797856271267,
  0.011046118102967739,
  0.008528106845915318,
  -0.05918348208069801,
  0.027340078726410866,
  -0.04205673187971115,
  0.05871699005365372,
  0.015138362534344196,
  -0.03152904286980629,
  -0.0013396431459113955,
  0.027405358850955963,
  0.024371469393372536,
  0.0013865812215954065,
  0.010173519141972065,
  -0.022959992289543152,
  -0.005069850478321314,
  0.01326177641749382,
  0.0393807515501976,
  0.07599280774593353,
  -0.08241260051727295,
  -0.007301719859242439,
  0.0590067058801651,
  0.021999960765242577,
  -0.0008191353990696371,
  0.00900810956954956,
  -0.004517426248639822,
  -0.03041398897767067,
  -0.03365267813205719,
  0.01974521204829216,
  -0.07396765798330307,
  0.0029045536648482084,
  -0.012916609644889832,
  0.04320052638649

**OBSERVATIONS**:


---
- The embeddings look good. Dimension of vector is 768 for document chunks with chunk size=500
- Length of each embedding is same

### Vector Database

In [30]:
out_dir_2000 = '/content/drive/My Drive/AIML/Project5/medical_db_2000'
out_dir_1000 = '/content/drive/My Drive/AIML/Project5/medical_db_1000'
out_dir_500 = '/content/drive/My Drive/AIML/Project5/medical_db_500'

if not os.path.exists(out_dir_2000):
  os.makedirs(out_dir_2000)
if not os.path.exists(out_dir_1000):
  os.makedirs(out_dir_1000)
if not os.path.exists(out_dir_500):
  os.makedirs(out_dir_500)

In [31]:
vector_db_2000 = Chroma.from_documents(
    documents = document_chunks_2000,
    embedding = embedding_model,
    persist_directory=out_dir_2000
)

vector_db_1000 = Chroma.from_documents(
    documents = document_chunks_1000,
    embedding = embedding_model,
    persist_directory=out_dir_1000
)

vector_db_500 = Chroma.from_documents(
    documents = document_chunks_2000,
    embedding = embedding_model,
    persist_directory=out_dir_2000
)

In [32]:
vectorstore_2000 = Chroma(persist_directory=out_dir_2000,embedding_function=embedding_model)
vectorstore_1000 = Chroma(persist_directory=out_dir_1000,embedding_function=embedding_model)
vectorstore_500 = Chroma(persist_directory=out_dir_500,embedding_function=embedding_model)

/tmp/ipython-input-1304466900.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore_2000 = Chroma(persist_directory=out_dir_2000,embedding_function=embedding_model)


In [33]:
vectorstore_2000.embeddings

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': True, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': True, 'pooling_mode_mean_tokens': False, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='BAAI/bge-base-en-v1.5', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [34]:
vectorstore_1000.embeddings

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': True, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': True, 'pooling_mode_mean_tokens': False, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='BAAI/bge-base-en-v1.5', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [35]:
vectorstore_500.embeddings

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': True, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': True, 'pooling_mode_mean_tokens': False, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='BAAI/bge-base-en-v1.5', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [36]:
vectorstore_2000.similarity_search("Sepsis in critical care",k=3)

[Document(metadata={'format': 'PDF 1.7', 'file_path': '/content/drive/My Drive/AIML/Project5/medical_diagnosis_manual.pdf', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'keywords': '', 'page': 2453, 'modDate': 'D:20251115205043Z', 'source': '/content/drive/My Drive/AIML/Project5/medical_diagnosis_manual.pdf', 'author': '', 'creationdate': '2012-06-15T05:44:40+00:00', 'moddate': '2025-11-15T20:50:43+00:00', 'trapped': '', 'creator': 'Atop CHM to PDF Converter', 'creationDate': 'D:20120615054440Z', 'subject': '', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'total_pages': 4114}, page_content='Chapter 227. Sepsis and Septic Shock\nIntroduction\n(See also Ch. 226.)\nSepsis, severe sepsis, and septic shock are inflammatory states resulting from the systemic\nresponse to bacterial infection. In severe sepsis and septic shock, there is critical reduction in\ntissue perfusion. Common causes include gram-negative organisms, staphylococci, and\nmeningococci. S

**OBSERVATIONS**:


---
- Similarity search returned info on our 'sepsis in critical care' search on 3 references since k=3 : Two references on: page 2453 and one on: page 2400


### Retriever

In [37]:
retriever_2000 = vectorstore_2000.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3}
)

In [38]:
retriever_1000 = vectorstore_1000.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3}
)

In [39]:
retriever_500 = vectorstore_500.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3}
)

In [40]:
rel_docs = retriever_2000.get_relevant_documents(query="What is the protocol for managing sepsis in a critical care unit?", k=3)
rel_docs

/tmp/ipython-input-2641204433.py:1: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  rel_docs = retriever_2000.get_relevant_documents(query="What is the protocol for managing sepsis in a critical care unit?", k=3)


[Document(metadata={'subject': '', 'creationdate': '2012-06-15T05:44:40+00:00', 'file_path': '/content/drive/My Drive/AIML/Project5/medical_diagnosis_manual.pdf', 'moddate': '2025-11-15T20:50:43+00:00', 'source': '/content/drive/My Drive/AIML/Project5/medical_diagnosis_manual.pdf', 'keywords': '', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'modDate': 'D:20251115205043Z', 'format': 'PDF 1.7', 'trapped': '', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'page': 2400, 'author': '', 'total_pages': 4114, 'creator': 'Atop CHM to PDF Converter', 'creationDate': 'D:20120615054440Z'}, page_content="16 - Critical Care Medicine\nChapter 222. Approach to the Critically Ill Patient\nIntroduction\nCritical care medicine specializes in caring for the most seriously ill patients. These patients are best\ntreated in an ICU staffed by experienced personnel. Some hospitals maintain separate units for special\npopulations (eg, cardiac, surgical, neurologic, pediatric, 

In [42]:
model_output = llm(
      user_input,
      max_tokens=2000,
      temperature=0.2,
    )

Llama.generate: prefix-match hit


In [43]:
model_output['choices'][0]['text']

"\n\nA fractured leg is a serious injury that requires prompt medical attention. Here are some necessary precautions and treatment steps for a person who has sustained a leg fracture while hiking:\n\n1. Assess the situation: If you or someone in your group has sustained a leg fracture, assess the scene to ensure everyone's safety. Check for any signs of other injuries, such as head trauma or bleeding. Try to keep the injured person calm and still to prevent further injury.\n2. Call for help: If possible, call for emergency medical assistance using a cell phone or satellite phone. Provide the location and nature of the injury to the dispatcher. If there is no reception, send someone in your group to seek help while others stay with the injured person.\n3. Immobilize the leg: Use a splint, a makeshift splint, or a hiking pole to immobilize the fractured leg. Be careful not to move the leg excessively as it can cause further damage. If you don't have any materials to make a splint, try to

**OBSERVATIONS**:


---

- The model is giving relevant responses based on our settings for Sepsis query in ~2000 max_tokens

### System and User Prompt Template

In [44]:
qna_system_message = system_prompt

In [45]:
qna_user_message_template = """
Context:
{context}

Question:
{question}
"""

### Response Function

In [46]:
def generate_rag_response(user_input, chunk_size=500, k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    if chunk_size==2000:
        retriever = retriever_2000
    elif chunk_size==1000:
        retriever = retriever_1000
    else:
        retriever = retriever_500


    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    prompt = qna_system_message + '\n' + user_message

    # Generate the response
    try:
        response = llm(
                  prompt=prompt,
                  max_tokens=max_tokens,
                  temperature=temperature,
                  top_p=top_p,
                  top_k=top_k
                  )

        # Extract and print the model's response
        response = response['choices'][0]['text'].strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

**OBSERVATIONS**:


---

- We have defined the functions and default parameters we can use for RAG


## Question Answering using RAG

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [47]:
user_input = "What is the protocol for managing sepsis in a critical care unit?"
print(generate_rag_response(user_input,chunk_size=500, k=3,max_tokens=500,temperature=0,top_p=0.95,top_k=50))

Llama.generate: prefix-match hit


Answer:
Sepsis is a life-threatening condition caused by the body's response to infection. In a critical care unit, managing sepsis involves several steps to ensure optimal patient outcomes. Here are some key aspects of sepsis management in a critical care setting:

1. Early recognition and diagnosis: Recognize sepsis early based on clinical signs such as fever, tachycardia, respiratory distress, and altered mental status. Use the Sequential Organ Failure Assessment (SOFA) score to assess organ dysfunction.

2. Fluid resuscitation: Administer intravenous fluids to maintain adequate blood pressure and tissue perfusion. Aim for a mean arterial pressure (MAP) of 65-70 mmHg and a central venous oxygen saturation (ScvO2) of >70%.

3. Antibiotic therapy: Initiate broad-spectrum antibiotics as soon as possible based on the suspected infection source. Adjust the antibiotic regimen based on culture results and sensitivity data.

4. Vasopressor support: If fluid resuscitation fails to maintain a

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [48]:
user_input_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
print(generate_rag_response(user_input_2,chunk_size=500, k=3,max_tokens=500,temperature=0,top_p=0.95,top_k=50))

Llama.generate: prefix-match hit


Answer:
Appendicitis is a medical condition characterized by inflammation of the appendix. Common symptoms include:
1. Sudden pain in the lower right abdomen, often starting around the navel and then moving to the right side.
2. Loss of appetite.
3. Nausea and vomiting.
4. Fever (usually above 37.5°C or 99.5°F).
5. Abdominal swelling and rigidity.
6. Constipation or diarrhea.

Appendicitis cannot be cured via medicine alone, as the inflammation can lead to appendix rupture if left untreated. In such cases, a surgical procedure called an appendectomy is required. This involves removing the appendix through either an open incision or laparoscopically. The choice of procedure depends on factors like the patient's overall health, the severity of the condition, and the surgeon's preference.

In summary, common symptoms for appendicitis include abdominal pain, loss of appetite, nausea, fever, abdominal swelling, and constipation or diarrhea. Appendicitis cannot be cured via medicine alone an

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [49]:
user_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
print(generate_rag_response(user_input_3,chunk_size=500, k=3,max_tokens=400,temperature=0,top_p=0.98,top_k=50))

Llama.generate: prefix-match hit


Answer:
Sudden patchy hair loss, also known as alopecia areata, is an autoimmune disorder that results in hair loss from some or all areas of the scalp. The exact cause of this condition is unknown, but it's believed to be related to a problem with the immune system.

Effective treatments for addressing sudden patchy hair loss include:
1. Corticosteroids: These medications can help reduce inflammation and suppress the immune response that causes hair loss. They are often administered through injections, topical applications, or oral medication.
2. Minoxidil: This medication is applied directly to the scalp and stimulates hair growth by increasing blood flow to the follicles. It's available over-the-counter as a liquid or foam.
3. DHT blockers: Finasteride and dutasteride are oral medications that block the production of dihydrotestosterone (DHT), which is believed to contribute to hair loss. They are primarily used for treating male pattern baldness but can also be effective for alopec

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [50]:
user_input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
print(generate_rag_response(user_input_4,chunk_size=2000, k=3,max_tokens=500,temperature=0,top_p=0.96,top_k=70))

Llama.generate: prefix-match hit


Answer:
The initial treatment for a person with a traumatic brain injury (TBI) involves ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure. Surgery may be needed in cases with more severe injuries to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas. In the first few days after the injury, maintaining adequate brain perfusion and oxygenation and preventing complications of altered sensorium are crucial. Subsequently, many patients require rehabilitation. The Merck Manual suggests that TBI is a common cause of death and disability worldwide, with causes including motor vehicle crashes, falls, assaults, and sports activities. The pathology of TBI can result in gross or microscopic structural changes depending on the mechanism and forces involved. Clinical manifestations vary greatly in severity and consequences, and injuries are commonly catego

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [51]:
user_input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery"
print(generate_rag_response(user_input_5,chunk_size=1000, k=3,max_tokens=500,temperature=0.2,top_p=0.95,top_k=50))

Llama.generate: prefix-match hit


Answer:
The Merck Manual suggests that for a person with a fractured leg due to a hiking accident, the following steps should be taken:
1. Immobilize the affected limb using a splint or cast to prevent further damage and promote healing. The extremity should be positioned in a functional position and elevated.
2. Examine and clean the wound daily, covering it with a sterile dressing.
3. Evaluate joint motion, muscle strength, sensation, and limb girth within 2 days after the injury to prevent contractures. Prevent immobilization by interrupting it with frequent periods of gentle exercise, progressing from passive to active.
4. For snakebites in the area, follow the guidelines provided for adjunctive measures, including tetanus prophylaxis and wound care.
5. If there is evidence of infection, administer antibiotics based on culture and sensitivity results. Subsequent antibiotic choices should be determined by these results.
6. For severe cases with compartment syndrome, consider fasciot

**OBSERVATIONS**:


---
- The quality of the responses is much better and we can observe the variations based on the parameters we give.
- I have selected larger chunks where the possibility of information on the topic in the manual will be larger, for eg, brain injury might have more information and references (chunk size=2000 may be required for context) while hairloss might have less references (chunk size=500 may be enough for context).
- Similarly I have varied the other parameters accordingly.
- In the next section, we can do more fine tuning based on the responses we got.


### Fine-tuning

#### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [52]:
user_input = "What is the protocol for managing sepsis in a critical care unit?"
print(generate_rag_response(user_input,chunk_size=1000, k=3,max_tokens=2500,temperature=0,top_p=0.90,top_k=50))

Llama.generate: prefix-match hit


Answer:
Sepsis is a life-threatening condition that requires prompt recognition and intervention. In a critical care unit, the following steps are typically taken to manage sepsis:
1. Early recognition: Identify sepsis suspects based on clinical signs such as fever, tachycardia, respiratory distress, altered mental status, and lactic acidosis.
2. Immediate fluid resuscitation: Administer intravenous fluids to maintain adequate tissue perfusion and organ function. The goal is to achieve a mean arterial pressure (MAP) of 65 mmHg or higher and a central venous oxygen saturation (ScvO2) of 70% or higher.
3. Antibiotic administration: Start broad-spectrum antibiotics as soon as possible based on the suspected source of infection and local microbiology data.
4. Source control: Identify and address the source of infection, such as removing an infected catheter or draining an abscess.
5. Vasopressor support: If fluid resuscitation fails to maintain adequate tissue perfusion, administer vasopre

#### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [53]:
user_input_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
print(generate_rag_response(user_input_2,chunk_size=1000, k=3,max_tokens=1000,temperature=0,top_p=0.95,top_k=50))

Llama.generate: prefix-match hit


Answer:
The common symptoms for appendicitis include abdominal pain, usually starting around the navel and then moving to the lower right side of the abdomen. Other symptoms may include loss of appetite, nausea, vomiting, fever, and a feeling of being sick or unwell in general. Appendicitis cannot be cured via medicine alone as it requires surgical removal of the appendix due to the risk of rupture and potential complications such as peritonitis. The surgical procedure for treating appendicitis is typically an appendectomy, which can be performed either open or laparoscopically. Before the surgery, the patient is usually


#### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [54]:
user_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
print(generate_rag_response(user_input_3,chunk_size=1000, k=3,max_tokens=1000,temperature=0.3,top_p=0.98,top_k=50))

Llama.generate: prefix-match hit


Answer:
Sudden patchy hair loss, also known as alopecia areata, is a common condition characterized by round or oval bald patches on the scalp. The exact cause of this autoimmune disorder is unknown, but it's believed to be triggered by a combination of genetic and environmental factors.

Effective treatments for addressing sudden patchy hair loss include:
1. Topical corticosteroids: These medications can help stimulate hair regrowth by reducing inflammation in the affected area. They are typically applied directly to the bald spot using a small brush or cotton swab.
2. Immunotherapy: This treatment involves applying contact sensitizers like diphencyprone or squaric acid dibutylester to the bald spot to stimulate an immune response and promote hair regrowth.
3. Systemic corticosteroids: In severe cases, oral corticosteroids may be prescribed to help reduce inflammation and promote hair growth. However, they come with potential side effects like weight gain, mood swings, and increased r

#### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [55]:
user_input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
print(generate_rag_response(user_input_4,chunk_size=2000, k=3,max_tokens=2000,temperature=0,top_p=0.96,top_k=70))

Llama.generate: prefix-match hit


Answer:
The initial treatment for a person with a traumatic brain injury (TBI) involves ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure. Surgery may be needed in cases with more severe injuries to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas. In the first few days after the injury, maintaining adequate brain perfusion and oxygenation and preventing complications of altered sensorium are crucial. Subsequently, many patients require rehabilitation. The Merck Manual suggests that TBI is a common cause of death and disability worldwide, with causes including motor vehicle crashes, falls, assaults, and sports activities. The pathology of TBI can result in gross or microscopic structural changes depending on the mechanism and forces involved. Clinical manifestations vary greatly in severity and consequences, and injuries are commonly catego

#### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [56]:
user_input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
print(generate_rag_response(user_input_5,chunk_size=1000, k=3,max_tokens=2000,temperature=0.2,top_p=0.95,top_k=40))

Llama.generate: prefix-match hit


Answer:
For a person who has fractured their leg during a hiking trip, the following precautions and treatment steps should be taken:

1. Immobilize the affected limb using a splint or cast to prevent further damage and promote healing. Ensure that the splint or cast is not too tight, causing discomfort or restricting blood flow.
2. Apply ice packs intermittently to reduce swelling and pain. Be sure not to apply ice directly to the skin for extended periods as it can cause frostbite.
3. Elevate the affected limb above heart level whenever possible to minimize swelling and improve circulation.
4. Monitor vital signs, including pulse, respiration rate, and blood pressure, to ensure that the person remains stable and does not experience complications such as shock or excessive bleeding.
5. Administer pain medication as prescribed by a healthcare professional to manage discomfort and help the person rest more comfortably.
6. Provide tetanus prophylaxis if necessary, based on the person's v

**OBSERVATIONS**:


---
- We have fine tuned the responses further by varying the parameters to generate RAG response.
- Visually, the responses look good for information purposes for medical professionals.
- But if it is actually correct can be verified using groundedness and relevance in the next section


## Output Evaluation

Let us now use the LLM-as-a-judge method to check the quality of the RAG system on two parameters - retrieval and generation. We illustrate this evaluation based on the answeres generated to the question from the previous section.

- We are using the same Mistral model for evaluation, so basically here the llm is rating itself on how well he has performed in the task.

In [57]:
groundedness_rater_system_message  = """
You are an AI assistant evaluating how well an AI-generated answer aligns with a given context. Your task is to rate the answer's groundedness, ensuring it only uses information explicitly present in the provided text.

Evaluation Criteria:
Rate the groundedness on a scale of 1 to 5:
• 1 (Not Grounded): The answer is entirely unsupported by the context or directly contradicts it.
• 2 (Poorly Grounded): The answer contains significant factual errors or makes claims not found in the context, with minimal alignment.
• 3 (Partially Grounded): The answer includes some accurate information from the context but also contains unsupported claims or extrapolations.
• 4 (Mostly Grounded): The answer is largely supported by the context, with only minor embellishments or slight omissions of detail.
• 5 (Appropriately Grounded): Every claim in the answer is directly traceable and fully supported by the provided context.

Response Format: Please output the evaluation in the following format:
Score: [1-5]
Reasoning: [Brief explanation for the score based on specific evidence or discrepancies.]
"""

In [70]:
relevance_rater_system_message = """
You are an AI assistant, your role is to evaluate the relevance of an AI-generated answer to a given user question within its provided context. Assess how well the answer addresses the user's intent.

Evaluation Criteria:
Rate the relevance on a scale of 1 to 5:
• 1 (Irrelevant): The answer does not address the question at all or is entirely off-topic.
• 2 (Low Relevance): The answer touches on keywords but fails to answer the core question.
• 3 (Moderate Relevance): The answer partially addresses the question but misses significant aspects or details.
• 4 (High Relevance): The answer mostly addresses the question, with minor omissions or slightly off-topic details.
• 5 (Perfect Relevance): The answer fully and accurately addresses all aspects of the question and user intent, and is concise.

Response Format: Please output the evaluation in the following format:
Score: [1-5]
Reasoning: [Brief explanation for the score, referencing the question, context, and answer.]
"""

In [59]:
user_message_template = """
###Question
{question}

###Context
{context}

###Answer
{answer}
"""

In [61]:
llama_llm = Llama(
    model_path=model_path_llama,
    n_ctx=4096,
    n_gpu_layers=32,
    n_batch = 512
)

AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


In [71]:
def generate_ground_relevance_response(user_input,chunk_size=500,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    if chunk_size==2000:
        retriever = retriever_2000
    elif chunk_size==1000:
        retriever = retriever_1000
    else:
        retriever = retriever_500

    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=3)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)

    # Combine user_prompt and system_message to create the prompt
    prompt = f"""[INST]{qna_system_message}\n
                {'user'}: {qna_user_message_template.format(context=context_for_query, question=user_input)}
                [/INST]"""

    response = llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    answer =  response["choices"][0]["text"]

    # Combine user_prompt and system_message to create the prompt
    groundedness_prompt = f"""[INST]{groundedness_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    # Combine user_prompt and system_message to create the prompt
    relevance_prompt = f"""[INST]{relevance_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    response_1 = llama_llm(
            prompt=groundedness_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    response_2 = llama_llm(
            prompt=relevance_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    return response_1['choices'][0]['text'],response_2['choices'][0]['text']

OBSERVATIONS:


---


- Here We have used a different llm for verification of groundedness and relevance: llama_llm. The response is generated with our std llm which we have used through this exercise: llm (mistral)

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [72]:
ground,rel = generate_ground_relevance_response(user_input="What is the protocol for managing sepsis in a critical care unit?",chunk_size=1000, k=3,max_tokens=2500,temperature=0,top_p=0.90,top_k=50)

print("GROUNDEDNESS:\n",ground,end="\n\n************************\n")
print("RELEVANCE:\n",rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


GROUNDEDNESS:
   Score: 5 (Excellently Grounded)
The answer provides a comprehensive and accurate overview of the protocol for managing sepsis in a critical care unit, based on the provided text from The Merck Manual of Diagnosis & Therapy. The answer covers all the key points mentioned in the evaluation criteria, including early recognition, prompt initiation of treatment, fluid resuscitation, antibiotic therapy, source control, vasopressor support, corticosteroids, other supportive measures, close monitoring, early goal-directed therapy, and a multidisciplinary team approach. The answer also provides relevant explanations for each point, demonstrating a deep understanding of the material. Therefore, the answer is appropriately grounded and earns a score of 5.

************************
RELEVANCE:
   Score: 5 (Relevant)

The answer provided in the evaluation criteria addresses the user's question and provides relevant information on managing sepsis in a critical care unit. The answer c

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [73]:
ground,rel = generate_ground_relevance_response(user_input="What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?",chunk_size=1000, k=3,max_tokens=1000,temperature=0,top_p=0.95,top_k=50)

print("GROUNDEDNESS:\n",ground,end="\n\n************************\n")
print("RELEVANCE:\n",rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


GROUNDEDNESS:
   • Score: 5 (Excellently Grounded)
The answer provides a detailed and accurate description of the symptoms and treatment options for appendicitis, including the grounding information requested. The response format is also well-structured and easy to understand.





















































************************
RELEVANCE:
   • Relevance: 5 (Moderate relevance)
The answer provides a detailed and accurate description of the symptoms and surgical treatment for appendicitis, including the importance of prompt surgical intervention to prevent mortality. However, it does not directly address the user's question regarding the relevance of antibiotics in treating appendicitis. Therefore, I score it 5 (Moderate relevance).


### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [74]:
ground,rel = generate_ground_relevance_response(user_input="What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?",chunk_size=1000, k=3,max_tokens=1000,temperature=0.3,top_p=0.98,top_k=50)

print("GROUNDEDNESS:\n",ground,end="\n\n************************\n")
print("RELEVANCE:\n",rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


GROUNDEDNESS:
   Score: 4 (Mostly Grounded)
Reasoning: The answer provides a detailed and accurate evaluation of the given context, aligning with the criteria provided. The response highlights the various treatment options available for addressing alopecia areata, including topical, intralesional, systemic corticosteroids, and surgical options. Additionally, the explanation mentions the possible causes of sudden patchy hair loss, such as genetic susceptibility, environmental triggers, autoimmune response, concomitant disorders, and certain medications. The answer also acknowledges that a scalp biopsy or microscopic hair examination may be required for definitive diagnosis in some cases. Overall, the response demonstrates a thorough understanding of the topic and provides evidence-based information to support its claims.

************************
RELEVANCE:
   • Relevance Score: 4 (Perfect Relevance)
The answer provides a detailed evaluation of the user's question, addressing all aspect

### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [75]:
ground,rel = generate_ground_relevance_response(user_input="What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?",chunk_size=2000, k=3,max_tokens=2000,temperature=0,top_p=0.96,top_k=70)

print("GROUNDEDNESS:\n",ground,end="\n\n************************\n")
print("RELEVANCE:\n",rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


GROUNDEDNESS:
   Score: 4 (Appropriately Grounded)
Reasoning: The answer is grounded in the provided text as it accurately reflects the information provided in the chapter on traumatic brain injury from The Merck Manual of Diagnosis & Therapy. The answer highlights the initial treatment for a person with TBI, which includes ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure, and may require surgery for more severe injuries. It also mentions the importance of maintaining adequate brain perfusion and oxygenation in the first few days after the injury to prevent complications of altered sensorium, and the need for rehabilitation subsequently. The answer does not contain any factual errors or claim unsupported by the provided text.

************************
RELEVANCE:
   Score: 5 (Perfect Relevance)
The answer provided is directly related to the user's question and provides a detailed explanation of the initial treatment for a person with a tra

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [76]:
ground,rel = generate_ground_relevance_response(user_input="What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?",chunk_size=1000, k=3,max_tokens=2000,temperature=0.2,top_p=0.95,top_k=40)

print("GROUNDEDNESS:\n",ground,end="\n\n************************\n")
print("RELEVANCE:\n",rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


GROUNDEDNESS:
   Score: 5 (Appropriately Grounded)
Reasoning: The answer provides a comprehensive and accurate overview of the necessary precautions and treatment steps for a person with a fractured leg during a hiking trip, entirely supported by the provided context. The answer covers various aspects of care and recovery, including immobilization, pain management, infection prevention, exercise, and rehabilitation. It also acknowledges the importance of seeking medical attention if signs of infection occur.
The answer does not contain any significant factual errors or make claims not found in the context. The information provided is directly traceable to the given text, with clear explanations and supporting details. The answer demonstrates a good understanding of the topic and provides appropriate guidance for personal use.
Overall, the answer is well-grounded and appropriately aligned with the provided context, making it an excellent response to the question.

**********************

**OBSERVATIONS**:


---

Groundedness:
  - Since our groundedness scores are 4 to 5 which are ideal, hence our model is performing very well in generating the responses which stick to the actual data provided to formulate responses

Relevance:
  - Since the relevance scores are also 4 to 5 for all 5 of our queries, hence our responses are also relevant to the question asked. This helps verify if we are not aware of the technicality of the material.

Reading through the response gives more explanation and understanding on the groundedness as well as the relevance


## Actionable Insights and Business Recommendations

#### **ACTIONABLE INSIGHTS**:


---
Following observations were made as we navigate through the project

1. Model:
 - Selecting the model and using it affects how you design your notebook. Different models will require different config, setup and coding. Hence, it was difficult to switch to a random model from huggingface and use the same colab notebook.
2. Defining Parameters:
 - Defining parameters and system prompt appropriately helps in tweaking the model to output the desired formats. Tweaking the parameters is important for the model to perform in an ideal way, like giving complete responses, not hallucinating, giving grounded and relevant responses.
3. Business knowledge and training:
 - Knowledge of the domain is important to know if the model is giving accurate responses, specially for critical domains like this healthcare one where patient safety and health can be affected by relying on inaccurate responses.
 - Hence we should have accurate and latest data for the model to be trained on and have a verification method to ensure that the output is accurate.








#### **BUSINESS RECOMMENDATIONS**


---
Following are a few business recommendations:

1. Knowledge of the training model:
- The Manual we had was a good one, however, in the real world, the
medical knowledge needs to be kept updated regularly based on new learnings. This should reflect in the database of our model's training set.
2.  Disclaimers:
- There has to be an understanding that while AI can provide information to the best of the technological ability, we cannot fully rely on it for the source of truth. Human discretion is needed. This needs to be communicated and set as disclaimers with the users.
3. Use cases:
- This type of AI solution is highly useful in remote and less advanced places in the world. With the help of internet, this can be an effect use case in such places where without proper medical knowledge, the doctors can still use such a tool with limited resources to treat patients.
- It can also be useful for students for learning and in case of emergencies.
Such a model can further be trained in on top of other language models - to be globally accessible in different global spoken languages.
4.  B2C:
- Once this model is used extensively and is mature enough to be relevant, it can also be used in a chat bot portal for patients or as an app as a B2C (Business to Consumer) usecase. However, the appropriate guidelines and limitations still apply.


